# 06 - Richer features: pawn structure, king safety, development, tactics

The model comparison in `05_model_comparison.ipynb` showed tree-based models only marginally beating logistic regression (~65.3% vs ~64.9% accuracy) — evidence the ~65% ceiling is coming from the *features*, not the model class. This notebook adds 10 new features to `extract_features_v2()` in [`src/features.py`](../src/features.py), all classical chess evaluation concepts we hadn't captured yet:

- **Pawn structure**: `doubled_pawns_diff`, `isolated_pawns_diff`, `passed_pawns_diff`
- **King safety**: `king_safety_diff` (attacked squares around the king), `pawn_shield_diff`
- **Piece activity/development**: `bishop_pair_diff`, `rook_open_files_diff`, `undeveloped_minors_diff`
- **Tactical exposure**: `hanging_pieces_diff` (attacked-and-undefended pieces)
- **Space**: `advanced_pawns_diff`

All are expressed as white-minus-black diffs, oriented so positive always means "better for white" — same convention as `material_diff`/`mobility_diff` from the original feature set.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.features import extract_features_v2, add_features_v2

train_df = pd.read_csv('../data/processed/train.csv', low_memory=False)
test_df = pd.read_csv('../data/processed/test.csv', low_memory=False)

train_df.shape, test_df.shape

((822936, 19), (208378, 19))

## Sanity check
Verify the new features on the after-1.e4 position: white's `e2` pawn just vacated its king-shield square, so `pawn_shield_diff` should go negative (white's shield is now weaker than black's intact one) while everything else stays at 0 (nothing else changed structurally yet).

In [2]:
after_e4_fen = train_df.loc[train_df['fen'].str.startswith('rnbqkbnr/pppppppp/8/8/4P3'), 'fen'].iloc[0]
new_feature_names = [
    'bishop_pair_diff', 'doubled_pawns_diff', 'isolated_pawns_diff', 'passed_pawns_diff',
    'king_safety_diff', 'pawn_shield_diff', 'rook_open_files_diff',
    'undeveloped_minors_diff', 'hanging_pieces_diff', 'advanced_pawns_diff',
]
feats = extract_features_v2(after_e4_fen)
{k: feats[k] for k in new_feature_names}

{'bishop_pair_diff': 0,
 'doubled_pawns_diff': 0,
 'isolated_pawns_diff': 0,
 'passed_pawns_diff': 0,
 'king_safety_diff': 0,
 'pawn_shield_diff': -1,
 'rook_open_files_diff': 0,
 'undeveloped_minors_diff': 0,
 'hanging_pieces_diff': 0,
 'advanced_pawns_diff': 0}

## Extract for train and test
Benchmarked at ~0.34ms/row (roughly 2.5x the original feature set — mostly the pawn-structure and hanging-pieces loops) — expect around 4.5min for train, 1.2min for test.

In [3]:
%%time
train_df = add_features_v2(train_df)
test_df = add_features_v2(test_df)

train_df.shape, test_df.shape

CPU times: user 5min 58s, sys: 3.72 s, total: 6min 1s
Wall time: 6min 2s


((822936, 53), (208378, 53))

## Check correlations with `white_win`
See whether any of the new features carry a stronger signal than what we already had (`material_diff` was 0.35, `mobility_diff` 0.25 in the original set).

In [4]:
train_df[new_feature_names + ['white_win']].corr()['white_win'].drop('white_win').sort_values(key=abs, ascending=False)

king_safety_diff           0.205719
passed_pawns_diff          0.188757
pawn_shield_diff           0.098221
hanging_pieces_diff        0.075473
advanced_pawns_diff        0.062555
rook_open_files_diff       0.060625
bishop_pair_diff           0.059327
isolated_pawns_diff        0.039752
undeveloped_minors_diff    0.033477
doubled_pawns_diff        -0.004159
Name: white_win, dtype: float64

## Save enriched splits

In [5]:
train_df.to_csv('../data/processed/train_features_v2.csv', index=False)
test_df.to_csv('../data/processed/test_features_v2.csv', index=False)
print('saved.')

saved.
